# Reusable Template: Time-Series / Aggregated Linear Regression (Python)

Copy this notebook when you need to:
- Aggregate a panel to a yearly (or period) mean,
- Fit a simple OLS trend,
- Project forward,
- Check robustness with noise / window / bootstrap simulations,
- Produce audience-adapted commentary.

**Swap only the data path, target column, and a few labels.**

## Configuration — edit these

In [ ]:
DATA_PATH      = "data/honeyproduction.csv"   # ← change
TIME_COL       = "year"
TARGET_COL     = "totalprod"                 # ← change
GROUP_AGG      = "mean"                      # mean | sum | median
FUTURE_START   = 2013
FUTURE_END     = 2050
NOISE_FRAC     = 0.15
N_SIMS         = 300
N_BOOT         = 500
RANDOM_SEED    = 42

## Imports & load

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

plt.style.use("seaborn-v0_8-whitegrid")

df = pd.read_csv(DATA_PATH)
print(df.shape)
display(df.head())

## Aggregate → X, y

In [ ]:
agg = df.groupby(TIME_COL)[TARGET_COL].agg(GROUP_AGG).reset_index()
X = agg[TIME_COL].values.reshape(-1, 1)
y = agg[TARGET_COL].values
print(agg.head())

## Fit + diagnostics

In [ ]:
regr = LinearRegression().fit(X, y)
print(f"Slope:     {regr.coef_[0]:.4g}")
print(f"Intercept: {regr.intercept_:.4g}")
print(f"R²:        {regr.score(X, y):.4f}")
y_pred = regr.predict(X)

## Visualise fit + future projection

In [ ]:
X_fut = np.arange(FUTURE_START, FUTURE_END + 1).reshape(-1, 1)
y_fut = regr.predict(X_fut)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(X, y, s=50, edgecolor="k", label="Observed")
ax.plot(X, y_pred, lw=2, label="OLS fit")
ax.plot(X_fut, y_fut, lw=2, ls="--", label=f"Projection → {FUTURE_END}")
ax.set_xlabel(TIME_COL)
ax.set_ylabel(TARGET_COL)
ax.legend()
plt.tight_layout()
plt.show()

print(f"{FUTURE_END} prediction: {y_fut[-1]:,.0f}")

## Quick robustness: noise simulation + bootstrap CI

In [ ]:
np.random.seed(RANDOM_SEED)
slopes = [LinearRegression().fit(X, y + np.random.normal(0, y.std()*NOISE_FRAC, len(y))).coef_[0]
          for _ in range(N_SIMS)]
print(f"Slope under noise — mean {np.mean(slopes):.0f}, 95% [{np.percentile(slopes,2.5):.0f}, {np.percentile(slopes,97.5):.0f}]")

preds = []
idx = np.arange(len(y))
for _ in range(N_BOOT):
    s = np.random.choice(idx, size=len(idx), replace=True)
    preds.append(LinearRegression().fit(X[s], y[s]).predict([[FUTURE_END]])[0])
lo, hi = np.percentile(preds, [2.5, 97.5])
print(f"{FUTURE_END} bootstrap 95% CI: [{lo:,.0f}, {hi:,.0f}]")

## Audience checklist (fill after every new project)

- [ ] Data literacy of primary reader? (high → keep R², CI; low → headline + one chart)
- [ ] Subject-matter expertise? (expert → skip definitions; novice → define every term)
- [ ] Executive skim? → put the single most important number in the first paragraph
- [ ] Technical supervisor? → appendix with code, residual plots, alternate estimators
- [ ] Mixed audience? → clear section headings so each reader can stop early